In [1]:
import os
import math
import random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
BASE = "/content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data/ibdts_processed_data"
VIDEO_DIR = os.path.join(BASE, "3D_Motion_Videos")
HR_DIR    = os.path.join(BASE, "hr")


In [4]:
N_FRAMES = 32
IMG_SIZE = (128, 128)    # height, width
STRIDE_SECONDS = 1.0     # slide by 1 second between clip starts (overlap)
BATCH_SIZE = 4
EPOCHS = 12
FPS_ASSUME = None        # if None we read FPS from each video; set to 30 if needed
SEED = 42
SHUFFLE = True

In [5]:
import os
import cv2

BASE = "/content/drive/MyDrive/HCI Project/Datasets/ibdts_processed_data/ibdts_processed_data"
VIDEO_DIR = os.path.join(BASE, "3D_Motion_Videos")
def get_video_info(path):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return {"fps": fps, "frames": frames}


In [6]:
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# -----------------------
# HELPERS: HR loading & interval mean
# -----------------------
def load_hr_csv(hr_path):
    """Load HR CSV into dataframe. Accepts CSVs that have:
       - a numeric HR column (named 'HR' or first numeric column), optionally
       - an optional 'time_sec' column with timestamps in seconds.
    """
    df = pd.read_csv(hr_path)
    # find HR column
    if 'HR' in df.columns:
        hr_col = 'HR'
    else:
        num_cols = df.select_dtypes(include='number').columns
        if len(num_cols) == 0:
            raise ValueError(f"No numeric columns found in {hr_path}")
        # prefer second numeric column if first is 'time_sec'
        if 'time_sec' in num_cols and len(num_cols) > 1:
            hr_col = [c for c in num_cols if c != 'time_sec'][0]
        else:
            hr_col = num_cols[0]
    return df, hr_col

def interval_mean_hr(hr_df, hr_col, start_sec, end_sec):
    """
    Compute mean HR over [start_sec, end_sec).
    If hr_df has 'time_sec' use rows inside the interval; otherwise treat hr_df as 1Hz samples (index -> seconds).
    """
    if 'time_sec' in hr_df.columns:
        # pick all HR rows whose time is in the interval
        window = hr_df[(hr_df['time_sec'] >= start_sec) & (hr_df['time_sec'] < end_sec)]
        if len(window) > 0:
            return float(window[hr_col].mean())
        # no samples inside interval -> interpolate at endpoints and average
        t = hr_df['time_sec'].values
        v = hr_df[hr_col].values
        # clamp query times to range to avoid extrapolation
        q_times = np.array([max(min(start_sec, t[-1]), t[0]), max(min(end_sec, t[-1]), t[0])])
        iq = np.interp(q_times, t, v)
        return float(iq.mean())
    else:
        # assume 1 value per second, index 0 -> second 0
        start_idx = int(math.floor(start_sec))
        end_idx = int(math.ceil(end_sec))
        start_idx = max(start_idx, 0)
        end_idx = min(end_idx, len(hr_df))
        if start_idx >= end_idx:
            return float(hr_df.iloc[-1][hr_col])  # fallback
        return float(hr_df.iloc[start_idx:end_idx][hr_col].mean())

def build_clip_index(video_dir, hr_dir, n_frames=N_FRAMES, stride_seconds=STRIDE_SECONDS):
    """
    Returns list of dicts for every possible sliding clip:
      {video_path, start_frame, end_frame, fps, clip_seconds, hr_mean, participant}
    """
    video_paths = sorted([p for p in Path(video_dir).glob("*.mp4")])
    records = []
    for v in video_paths:
        base = v.stem
        hr_path = os.path.join(hr_dir, f"{base}_hr.csv")
        if not os.path.exists(hr_path):
            print(f"Skipping {v.name}: missing HR CSV {Path(hr_path).name}")
            continue

        info = get_video_info(str(v))
        fps = info["fps"] if info["fps"] and info["fps"] > 0 else FPS_ASSUME
        if fps is None:
            raise RuntimeError(f"FPS unknown for {v.name}; set FPS_ASSUME or fix file.")
        total_frames = info["frames"]

        frames_per_clip = int(n_frames)  # directly use N_FRAMES
        stride_frames = max(1, int(round(stride_seconds * fps)))
        max_start = total_frames - frames_per_clip
        if max_start < 0:
            print(f"Video {v.name} too short ({total_frames} frames) for N_FRAMES={n_frames}, skipping.")
            continue

        hr_df, hr_col = load_hr_csv(hr_path)
        participant = base.split("_")[0]
        for start_f in range(0, max_start + 1, stride_frames):
            end_f = start_f + frames_per_clip
            start_sec = start_f / fps
            end_sec = end_f / fps
            hr_mean = interval_mean_hr(hr_df, hr_col, start_sec, end_sec)
            records.append({
                "video_path": str(v),
                "start_frame": int(start_f),
                "end_frame": int(end_f),
                "fps": float(fps),
                "clip_seconds": frames_per_clip / fps,
                "hr_mean": float(hr_mean),
                "participant": participant
            })
    return records

# Build index
clip_index = build_clip_index(VIDEO_DIR, HR_DIR, n_frames=N_FRAMES, stride_seconds=STRIDE_SECONDS)
print("Total clips found:", len(clip_index))
if len(clip_index) == 0:
    raise SystemExit("No clips found. Check VIDEO_DIR, HR_DIR, and N_FRAMES/FPS settings.")

# -----------------------
# Person-specific split
# -----------------------
participants = sorted(list({r['participant'] for r in clip_index}))
train_parts, rest = train_test_split(participants, test_size=0.3, random_state=SEED)
val_parts, test_parts = train_test_split(rest, test_size=0.5, random_state=SEED)

def filter_by_parts(records, parts_set):
    return [r for r in records if r['participant'] in parts_set]

train_records = filter_by_parts(clip_index, set(train_parts))
val_records   = filter_by_parts(clip_index, set(val_parts))
test_records  = filter_by_parts(clip_index, set(test_parts))

print(f"Participants total {len(participants)} -> train {len(train_parts)}, val {len(val_parts)}, test {len(test_parts)}")
print(f"Clip counts: train {len(train_records)}, val {len(val_records)}, test {len(test_records)}")



Total clips found: 2623
Participants total 20 -> train 14, val 3, test 3
Clip counts: train 1848, val 379, test 396


In [7]:
def sample_clip_from_video(video_path, start_frame, end_frame, n_frames=N_FRAMES, img_size=IMG_SIZE):
    cap = cv2.VideoCapture(video_path)
    if end_frame <= start_frame:
        idxs = np.array([start_frame] * n_frames, dtype=int)
    else:
        idxs = np.linspace(start_frame, end_frame - 1, num=n_frames, dtype=int)
    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok or frame is None:
            frame = np.zeros((img_size[0], img_size[1], 3), dtype=np.uint8)
        else:
            frame = cv2.resize(frame, (img_size[1], img_size[0]))
            frame = frame[..., ::-1]  # BGR -> RGB
        frames.append(frame.astype(np.float32) / 255.0)
    cap.release()
    return np.stack(frames, axis=0)  # (N_FRAMES, H, W, 3)

# -----------------------
# Generator yielding (clip, hr, participant)
# -----------------------
def clip_generator(records, shuffle=SHUFFLE):
    recs = list(records)
    if shuffle:
        random.shuffle(recs)
    for rec in recs:
        clip = sample_clip_from_video(rec['video_path'], rec['start_frame'], rec['end_frame'], n_frames=N_FRAMES, img_size=IMG_SIZE)
        yield clip.astype(np.float32), np.float32(rec['hr_mean']), rec['participant']

# -----------------------
# tf.data Datasets
# -----------------------
output_sig = (
    tf.TensorSpec(shape=(N_FRAMES, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.string)
)

def ds_from_records(records, batch_size=BATCH_SIZE, shuffle=SHUFFLE):
    gen = lambda: clip_generator(records, shuffle=shuffle)
    ds = tf.data.Dataset.from_generator(gen, output_signature=output_sig)
    # map to (x, y) for model
    ds_xy = ds.map(lambda x,y,p: (x,y), num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds_xy = ds_xy.shuffle(1024, seed=SEED)
    ds_xy = ds_xy.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds_xy

train_ds = ds_from_records(train_records, batch_size=BATCH_SIZE, shuffle=True)
val_ds   = ds_from_records(val_records,   batch_size=BATCH_SIZE, shuffle=False)
test_ds  = ds_from_records(test_records,  batch_size=BATCH_SIZE, shuffle=False)

# compute steps so fit won't complain
train_steps = max(1, math.ceil(len(train_records) / BATCH_SIZE))
val_steps = max(1, math.ceil(len(val_records) / BATCH_SIZE))
test_steps = max(1, math.ceil(len(test_records) / BATCH_SIZE))
print("Steps (train/val/test):", train_steps, val_steps, test_steps)

Steps (train/val/test): 462 95 99


In [ ]:
# CNN + LSTM model (drop-in replacement for 3D CNN)
from tensorflow.keras import layers, models, optimizers, losses, callbacks
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -----------------------
# Build CNN+LSTM model
# -----------------------
def build_cnn_lstm_model(n_frames=N_FRAMES, img_size=IMG_SIZE, cnn_embedding_dim=128, lstm_units=128):
    # Input shape: (N_FRAMES, H, W, 3)
    inp = layers.Input(shape=(n_frames, img_size[0], img_size[1], 3), name='video_clip')

    # small per-frame CNN
    def make_frame_cnn():
        f_in = layers.Input(shape=(img_size[0], img_size[1], 3))
        x = layers.Conv2D(32, 3, activation='relu', padding='same')(f_in)
        #x = layers.MaxPool2D(2)(x)
        #x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
        #x = layers.MaxPool2D(2)(x)
        #x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(cnn_embedding_dim, activation='relu')(x)
        return models.Model(f_in, x, name='frame_cnn')

    frame_cnn = make_frame_cnn()

    td = layers.TimeDistributed(frame_cnn)(inp)

    x = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=False))(td)

    #x = layers.Dense(128, activation='relu')(x)
    #x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1, activation='linear', name='hr')(x)

    model = models.Model(inputs=inp, outputs=out, name='cnn_lstm_hr')
    model.compile(optimizer=optimizers.Adam(1e-4), loss=losses.MeanSquaredError(), metrics=['mae'])
    return model

# Build the model
model = build_cnn_lstm_model(n_frames=N_FRAMES, img_size=IMG_SIZE, cnn_embedding_dim=32, lstm_units=32)
model.summary()

# -----------------------
# Callbacks & training
# -----------------------
cbs = [
    callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=train_steps,
    validation_steps=val_steps,
    callbacks=cbs,
    verbose=2
)

Model: "cnn_lstm_hr"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ video_clip (InputLayer)         │ (None, 32, 128, 128,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 32, 32)         │         1,952 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hr (Dense)                      │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,705 (80.88 KB)

 Trainable params: 20,705 (80.88 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/12


In [ ]:
# -----------------------
# Evaluate on test set (collect arrays)
# -----------------------
preds = []
trues = []
parts = []
for clip, hr, part in clip_generator(test_records, shuffle=False):
    p = model.predict(np.expand_dims(clip, axis=0), verbose=0)[0,0]
    preds.append(float(p))
    trues.append(float(hr))
    parts.append(part)

preds = np.array(preds)
trues = np.array(trues)

mae = mean_absolute_error(trues, preds)
rmse = np.sqrt(mean_squared_error(trues, preds))
r2 = r2_score(trues, preds) if len(trues)>1 else float('nan')
corr = np.corrcoef(trues, preds)[0,1] if len(trues)>1 else float('nan')

print("TEST METRICS (CNN+LSTM):")
print(f"MAE: {mae:.2f} bpm")
print(f"RMSE: {rmse:.2f} bpm")
print(f"R²: {r2:.3f}")
print(f"Pearson correlation: {corr:.3f}")

# Per-participant MAE (requires pandas)
import pandas as pd
dfp = pd.DataFrame({"participant": parts, "true": trues, "pred": preds})
per = dfp.groupby("participant").apply(lambda g: mean_absolute_error(g['true'], g['pred']))
print("Per-participant MAE (sorted):\n", per.sort_values())

# -----------------------
# Plots: training curves + scatter
# -----------------------
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('epoch'); plt.ylabel('MSE loss'); plt.legend(); plt.grid(True)
plt.subplot(1,2,2)
if 'mae' in history.history:
    plt.plot(history.history['mae'], label='train_mae')
    plt.plot(history.history['val_mae'], label='val_mae')
    plt.xlabel('epoch'); plt.ylabel('MAE'); plt.legend(); plt.grid(True)
plt.show()

plt.figure(figsize=(6,6))
plt.scatter(trues, preds, alpha=0.6)
mn, mx = min(trues.min(), preds.min()), max(trues.max(), preds.max())
plt.plot([mn,mx],[mn,mx], 'r--')
plt.xlabel("True HR"); plt.ylabel("Pred HR"); plt.title("Test: True vs Pred"); plt.grid(True)
plt.show()